# Monotonic and Bounded GP Conditioning

This notebook conditions a GP regression posterior to satisfy two shape constraints at once: every sampled function must be **non-decreasing**, and must stay between a lower bound $\ell(x) = 0$ and an upper bound $u(x) = \frac{1}{3}\ln(30x + 1) + 0.1$. Such constraints are awkward to build into a GP analytically, but here they are just conditions applied while sampling the flow. The data are 7 exact observations of

$$f(x) = \frac{1}{3}\left[\arctan(20x - 10) - \arctan(-10)\right].$$

We compare several ways of imposing the constraints: gradient-based guidance with an ODE or SDE sampler, gradient-free Ensemble Kalman guidance, and pCN correction.

## 1. Imports

In [ ]:
import sys
from pathlib import Path
from typing import Callable

import gpytorch
import torch

from flow_inference.conditions import BinaryCondition, ProductCondition
from flow_inference.flow_utils import build_grid_in_box
from flow_inference.flows import GaussianProcessFlow
from flow_inference.gp_utils import gp_posterior
from flow_inference.guidance import GuidanceEnsembleKalman, GuidanceMC
from flow_inference.sampler import IntegratorSampler, PCNCorrector

sys.path.insert(0, str(Path.cwd().parent))  # examples/, for plotting.py
from plotting import plot_paths

torch.set_default_dtype(torch.float64)

## 2. Configuration

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
N = 64               # grid points on [0, 1)
noise = 1e-10        # observation noise variance: the data are effectively exact
num_samples = 1_000

## 3. Target Function, Data and Bounds

The 7 observations sit at $x_i = 0.1 + 1/(i+1)$, $i = 1, \dots, 7$, so they cluster in $[0.22, 0.6]$ and leave both ends of the domain to the prior and the constraints. `plotter` overlays the observations and the two bounds on the usual band-plus-traces figure.

In [ ]:
def fun(x):
    return (torch.arctan(20.0 * x - 10.0) - torch.arctan(torch.tensor(-10.0))) / 3


def upper(x):
    return torch.log(30.0 * x + 1.0) / 3 + 0.1


def lower(x):
    return torch.zeros_like(x)


X = torch.tensor([[0.1 + 1 / (i + 1)] for i in range(1, 8)])  # (7, 1)
Y = fun(X).squeeze(-1)                                        # (7,)
X, Y = X.to(device), Y.to(device)


def plotter(samples, num_samples=10):
    grid = coords.squeeze(-1).cpu()  # the GP's own grid, so bounds line up with samples
    ax, _ = plot_paths(samples, num_samples=num_samples, x=grid)
    ax.scatter(X.squeeze(-1).cpu(), Y.cpu(), color="red", s=50, zorder=5)
    ax.plot(grid, upper(grid), color="black", linestyle="-.", label="Upper Bound")
    ax.plot(grid, lower(grid), color="black", linestyle="--", label="Lower Bound")
    ax.legend()

## 4. GP Posterior

An RBF kernel with lengthscale $0.1$ and outputscale $0.25$, set by hand rather than fitted. `gp_posterior` forms the conjugate posterior given the 7 observations; the flow evaluates its mean and kernel on the grid.

In [ ]:
coords = build_grid_in_box(N, 1, device)  # (N, 1)
kernel = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel()).to(device)
kernel.base_kernel.lengthscale = 0.1
kernel.outputscale = 0.25
p0_mean_fn, p0_kernel = gp_posterior(kernel, X, Y, noise=noise)

## 5. Unconditioned Samples

Samples from the GP posterior alone interpolate the data but are free to decrease and to leave the bounds away from the observations.

In [ ]:
flow = GaussianProcessFlow(p0_kernel, p0_mean_fn, coords, device=device)
sampler = IntegratorSampler(flow=flow)
samples = sampler.sample(num_samples, guidance=None, num_steps=1_000)
plotter(samples)

## 6. Shape Conditions

Both constraints are `BinaryCondition`s: `smooth_eval_condition` returns margins that should be positive, and the base class scores them with $\log\sigma(\text{margin}/v)$, a smooth relaxation of the hard constraint. Monotonicity uses the successive differences $f(x_{j+1}) - f(x_j)$; boundedness uses $u(x) - f(x)$ and $f(x) - \ell(x)$. `ProductCondition` asks for both.

In [ ]:
class MonotonicityCondition(BinaryCondition):
    """Non-decreasing, for a single-channel field on a 1D grid ordered left to right."""

    def smooth_eval_condition(self, x: torch.Tensor) -> torch.Tensor:
        f = x.squeeze(1)                           # (B, N)
        return (f[:, 1:] - f[:, :-1])[:, None, :]  # (B, 1, N-1)


class BoundedCondition(BinaryCondition):
    """Within [lower, upper] at each grid point, for a single-channel field."""

    def __init__(self, upper: Callable[[torch.Tensor], torch.Tensor],
                 lower: Callable[[torch.Tensor], torch.Tensor], grid: torch.Tensor,
                 v: float = 1e-6):
        super().__init__(v=v)
        self.register_buffer("upper_bound", upper(grid)[None, :])  # (1, N)
        self.register_buffer("lower_bound", lower(grid)[None, :])  # (1, N)

    def smooth_eval_condition(self, x: torch.Tensor) -> torch.Tensor:
        f = x.squeeze(1)  # (B, N)
        margins = [self.upper_bound - f, f - self.lower_bound]
        return torch.concat(margins, dim=-1)[:, None, :]  # (B, 1, 2N)


monotonic_condition = MonotonicityCondition(v=1e-6).to(device)
bounded_condition = BoundedCondition(upper, lower, coords.squeeze(-1), v=1e-6).to(device)
condition = ProductCondition([monotonic_condition, bounded_condition]).to(device)

## 7. Conditioned Sampling

The same condition can be imposed in several ways. Each subsection re-samples with it; all should give non-decreasing paths inside the bounds.

### 7.1 Gradient-based guidance (ODE)

`GuidanceMC` estimates the guidance term by differentiating the condition through Monte-Carlo draws of the clean sample.

In [ ]:
guidance = GuidanceMC(flow, condition, num_mc_samples=1).to(device)
samples = sampler.sample(num_samples, guidance=guidance, num_steps=1_000)
plotter(samples)

### 7.2 Gradient-based guidance (SDE)

The same guidance, integrating the stochastic rather than the deterministic dynamics.

In [ ]:
samples = sampler.sample(num_samples, guidance=guidance, num_steps=1_000, sde=True)
plotter(samples)

### 7.3 Gradient-free Ensemble Kalman guidance

`GuidanceEnsembleKalman` estimates the guidance with an Ensemble Kalman Inversion update, so it never differentiates the condition.

In [ ]:
enkf_guidance = GuidanceEnsembleKalman(flow, condition, num_mc_samples=10, num_iterations=1).to(device)
samples = sampler.sample(num_samples, guidance=enkf_guidance, num_steps=1_000, sde=True)
plotter(samples)

### 7.4 Ensemble Kalman guidance with pCN correction

Interleaving preconditioned Crank–Nicolson Metropolis steps corrects the remaining error in the guidance, allowing far fewer integration steps.

In [ ]:
corrector = PCNCorrector(flow, condition, num_steps=5, num_mc_samples=4).to(device)
samples = sampler.sample(num_samples, guidance=enkf_guidance, corrector=corrector, num_steps=250, sde=True)
plotter(samples)

### 7.5 pCN correction alone

With no guidance at all, the corrector does all the conditioning, using only `condition.logp`.

In [ ]:
samples = sampler.sample(num_samples, guidance=None, corrector=corrector, num_steps=500, sde=True)
plotter(samples)